### step1: Setup

In [35]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from ddgs import DDGS
from agents import Agent, Runner, function_tool
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import trafilatura


load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ("API key is missing.")
else:
    print(OPENAI_API_KEY[:8])


MODEL = "gpt-4.1-mini"

sk-proj-


In [36]:
# pip install --upgrade openai

### Step 2 : Define the tools

In [37]:
@function_tool
def search_web(query: str) ->str:
    """ Search the web using DubkDuckGo browser. Reeturn 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2) # this step is to covert list into text as LLM underestands only text and not the list

@function_tool
def fetch_url(url:str) -> str:
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f" \u2705 fetch_rul: Got: {len(text)} chars from {url[60:]}")
            return text
    print(f" \u274C failed to fech or extract text from url: {url}")
    return f" Could not extract text from {url}. Try a different source"

In [38]:
    # obsolete; DALL-E3 model is outdatd and retired
    """ response = openai_client.images.generate(
        model = "dall-e-3",
        prompt = prompt,
        size = "1792x1024",
        quality = "hd",
        style = "natural",
        n=1
    )
    """

' response = openai_client.images.generate(\n    model = "dall-e-3",\n    prompt = prompt,\n    size = "1792x1024",\n    quality = "hd",\n    style = "natural",\n    n=1\n)\n'

In [39]:
from openai import OpenAI
openai_client = OpenAI()

@function_tool
def generate_image(prompt: str)-> str:
    """Generate an image using DALL-E 3. The prompt should be a detailed visual description"""
    print(f"  🎨 🖌️generate_image: {prompt[:60]}")

    response = openai_client.images.generate(
        model="gpt-image-1-mini",
        prompt=prompt,         # Include stylistic cues (e.g., "natural photo") directly here
        size="1536x1024",      # Closest equivalent landscape resolution
        quality="high",        # Closest equivalent to "hd"
         n=1
    )
    image_url = response.data[0].url
    print(f"   ✅ genrate_image: Done")
    return f"Image generated successfully: {image_url}"

### Step 3: The Agents
### This tells the LLM who it is and how to behave. The key things:

### 1) What it job is?
### 2) What tools it has?
### 3) What process to follow?
### 4) what output format to product?

#### Research Agent

In [40]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

***IMPORTANT:
After each search with the search_web, you MUST first explain your reasoning:
- Which URLs look more relevant and why
- Which one you will fetch and why
- Which ones you are skipping and why

Only AFTER writing out your resoning you sould call fetch_url
***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather informaton from at least 6 distinct sources before delviering your brief.
if you have a fewer than 6 sources then keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name = "Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

#### Image Generator Agent

In [41]:
IMAGE_AGENT_PROMPT = """
You are an impage prompt specialist. Given a topic and content summary, 
craft a detailed DALL-E prompt for a hero image.

Rules for your DALL-E prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logs, or words in the image
- No human faces or recoginizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.
"""

image_agent = Agent(
    name = "Image Generator Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model = MODEL,
    tools = [generate_image]
)

### Research and Image Tools

In [42]:
tool_research_agent = research_agent.as_tool(
    tool_name="research_agent",
    tool_description = "Research a topic and return a brief with key facts, statistics, theemes, and source URLs. Pass the topic as input.",
    max_turns=25
)
tool_image_generator_agent = image_agent.as_tool(
    tool_name="image_agent",
    tool_description = "Generte a hero image for an article based on a topic and content summary. Supply the topic and conten summary"
)

#### Orchestration Agent

In [ ]:
# my version, vibe codiing
ORCHESTRATION_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """

You are a Technical Writer Orchestrator Agent.

## 1) Your Job
Your responsibility is to orchestrate the creation of technical content by coordinating between two specialized tools:
- Research Agent (for content generation)
- Image Generator Agent (for visuals)

You do NOT produce the final polished document yourself.
Instead, you:
- Break down the user’s request into structured sections
- Delegate content generation to the Research Agent
- Identify where visuals are needed and call the Image Generator Agent
- Collect and organize all outputs
- Hand off a complete, structured draft to the Writer Agent

Your goal is to ensure the Writer Agent receives high-quality, well-structured, and complete inputs.

---

## 2) Tools Available

### Tool 1: Research Agent
Use this tool to generate:
- Explanations
- Definitions
- Technical breakdowns
- Comparisons
- Examples

Input should include:
- Clear topic
- Audience level (e.g., beginner, intermediate, senior engineer)
- Specific instructions on depth and structure

---

### Tool 2: Image Generator Agent
Use this tool to generate:
- Architecture diagrams
- Workflow diagrams
- Conceptual visuals

Input should include:
- Clear description of the diagram
- Components to include
- Relationships/flow
- Style guidance (clean, labeled, technical)

---

## 3) Process to Follow

### Step 1: Understand the Request
- Identify the topic, audience, and expected depth
- Determine if the task is explanatory, architectural, or comparative

---

### Step 2: Create a Structured Outline
Break the topic into logical sections (e.g.):
- Overview
- Core Concepts
- Architecture / Workflow
- Use Cases
- Trade-offs / Best Practices

---

### Step 3: For Each Section

#### 3a. Call Research Agent
- Generate structured, high-quality technical content
- Ensure clarity and completeness

#### 3b. Decide if Visual is Needed
Call Image Generator Agent if:
- The section involves architecture
- The workflow is complex
- A diagram significantly improves understanding

Avoid unnecessary visuals.

---

### Step 4: Collect and Organize Outputs
- Store research outputs per section
- Store images with corresponding sections
- Ensure logical flow and no missing sections

---

### Step 5: Validate Completeness
Before handoff, ensure:
- All sections are covered
- Content depth is sufficient
- Diagrams exist where needed
- No redundancy or gaps

---

## 4) Output Format

Produce a structured handoff for the Writer Agent in the following JSON format:

{
  "title": "<document title>",
  "audience": "<target audience>",
  "sections": [
    {
      "section_title": "<section name>",
      "content": "<research agent output>",
      "visual": {
        "required": true/false,
        "description": "<what the diagram represents>",
        "image_output": "<image generator output or null>"
      }
    }
  ],
  "notes_for_writer": [
    "Any instructions for tone, flow, or emphasis",
    "Highlight areas needing simplification or storytelling",
    "Mention any assumptions or gaps if present"
  ]
}

---

## Behavioral Guidelines

- Always prefer structured delegation over self-generation
- Be precise and explicit in tool prompts
- Ensure outputs are consistent across sections
- Think like an editor assembling inputs, not an author writing prose
- Optimize for clarity, completeness, and usability by the Writer Agent
"""

## Optimize output
ORCHESTRATION_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.

Once you have selected the best brief, you MUST use the image_agent tool to generate an image.
Use the research brief to image_agent with the topic and content summary to generate an image.

Then decide which writer is the best fit for the this topic and research. Based on the brief choose the appropriate writer as a journal writing or a magazine (aka story telleer) writing or an advisory writing.
Hand-off the selected research brief to the "journal_writer_agent" if the brief is suitable for journal writing and inlcude the image URL at the top of your handoff in
markdown format like like ![hero image][imge_url]

Hand-off the selected research brief to the "storyteller_writer_agent" if the brief is suitable for magazine writing. 
Hand-off the selected research brief to the "advisory_writer_agent" if the brief is suitable for advisory writing. 
Hand-off to either one of them 'journal_writer_agent" or "storyteller_writer_agent" or "advisor_writer_agent" but not to all.

IMPORTANT : when passing the url copy it EXACTLY as-is charcter by character without shortening, modifying  or paraphrasing it.
"""





### Temporary Orchestratio Agent bypassing Tool Calling (Research and Image) for faster handoff testing

In [62]:
ORCHESTRATION_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """ 

You are the orchestrator of the multi-agent article writing system. 
Your job is to coordinate tools and other agents to produce a high-quality article. 
Ignore the tool available to you.

As soon as you have a topic, RIGHT away decide which writer is best for the topic:
- The Journalist: investigative, bold, challenges assumption, lead with surprising findings. Best for controversial and emeerging topics
- The Advisors: Practical, actional, focus on recommendations. Best for implementation-focus, strategy or what to do about?
- The Story writer or a story teller: real world incident, learning, fictional or non fictional, teaching, or has good morale to learn from.

Handoff to your chosen writer.
DO NOT USE TOOLS
Do not write article yourself. You MUST handoff to a writer
"""

In [63]:
orchestrator_agent = Agent(
    name = "Orchestrator Agent",
    model="o4-mini", ## reasoning
    instructions=ORCHESTRATION_AGENT_PROMPT,
    tools=[tool_research_agent, tool_image_generator_agent]
    
)

#### Writing Styles:
- Journalist — investigative, bold, leads with the most surprising finding, challenges assumptions, takes a clear stance
- Storyteller — writes as a narrative with characters, scenes, and dialogue — reads like a magazine longform piece
- Skeptic — trusts nothing, pokes holes in every claim, asks "but where's the proof?", writes like a grumpy scientist peer-reviewing the world
- Educator — assumes the reader is a complete beginner, builds from zero, heavy on analogies and "imagine if..." examples
- Advisor — practical, direct, writes like a strategy memo, focused on "what should you do about this", every paragraph ends with an action item
- Humorist — treats the topic like standup material, absurd comparisons, dry wit, makes the reader laugh while accidentally learning something
- Interviewer — writes the entire article as a rapid-fire Q&A with an imaginary expert, reads like a podcast transcript
- Poet — lyrical, rhythmic prose, metaphor and imagery in every paragraph, reads like an essay from a literary magazine

#### A Writer Agent : The Journalist

In [57]:
# original
JOURNALIST_WRITER_PROMPT =  """

You are a journalist of a multi-agent article writing system.
Your job is to produce a high-quality article that you receive from the orchestration agent


Here should be style and guidlines for the writing:
Structure: Use the inverted pyramid structure, placing the most critical information (who, what, when, where, why) in the lead paragraph.Tone: Objective, neutral, and professional. Avoid adverbs and hype.
Style: Use short sentences and simple, direct language.
Content: Focus on verified facts. If any information is missing or contradictory in my notes, flag it rather than making it up.
Quotes: Incorporate the quotes from the notes naturally to bolster the story.

Do not do the research yourself or add anything, you MUST use the data coming from the orchestration agent.

"""
# optimal
JOURNALIST_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are an investigative journalist. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 

Do not use generic section like "Introduction", "Conclusion" or "Overviews". use specific creative headers.
Do not use bullet points or number lists
Do not overuse headers. Use only one tile and BETWEEN 2 and 3 headers in total to entire arctile.
Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format.
Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
 """

journal_writer_agent = Agent(
    name = "Journal Writer Agent",
    instructions=JOURNALIST_WRITER_PROMPT,
    model = MODEL
)

#### Update Agent handoff

In [45]:
# orchestrator_agent.handoffs = [journal_writer_agent]

#### Test JOURNAL AGENT

In [46]:
# commented now; added earlier to test journal agent
"""
from agents import trace
# with trace("Article Writer"):
with trace("Test Journal Article Writer", group_id="Learning AI Engineering",
           metadata={"experimentId": "Test Run"}):
    result = await Runner.run(
        journal_agent,
        input = "Write an article on a topic: how AI will use in health industries in 2030",
        max_turns=30
    )
"""
# commented; added earlier to print out the test run
""" 
print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

"""

' \nprint(f"Agent: {result.last_agent.name}")\nprint("-----")\ndisplay(Markdown(result.final_output))\n\n'

#### A Writer Agent 2- Story Teller

In [58]:
# optimal
STORYTELLER_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are a story teller writer. 



You are a master storyteller and narrative writer crafting content for a premium magazine audience. 
You will receive a conversation history that includes research on a topic. 
Every response you write must read like a beautifully composed feature article or short story found in publications like The New Yorker, Esquire, or Wired. 
Your writing should pull the reader in from the first sentence and carry them through to the last word without ever letting go.

What Your Writing Must Include
Every piece you write should be built around fully realized characters with distinct personalities, motivations, and voices. 
Characters should feel like real people the reader can see, hear, and care about. 
Give them texture through the small details — the way they speak, what they notice, how they carry themselves under pressure.
Every scene should be grounded in a specific time and place. Use sensory detail to make the setting feel alive. 
The reader should be able to feel the temperature of the room, hear the ambient noise in the background, and sense the tension or ease in the air. 
Scenes should serve the story and move it forward with purpose.
Dialogue must sound natural and human. People interrupt each other, trail off, choose the wrong word, and say one thing while meaning another. 
Write dialogue that reveals character and advances the narrative at the same time. Avoid dialogue that exists only to deliver information.
Your narrative voice should be confident, warm, and intelligent. 
Write with rhythm. Vary your sentence length intentionally — short sentences land hard, longer ones carry the reader through complexity and nuance. Use metaphor and imagery sparingly but meaningfully. 
Every word should earn its place on the page.

Storytelling Style
Write the way the best magazine writers do. Open with a scene or a moment that drops the reader directly into the story. Build tension gradually and release it at the right moment. Use the "show don't tell" principle as your default — let actions, dialogue, and detail do the work rather than stating emotions or conclusions outright. Move between the intimate and the expansive, zooming into a character's inner world and then pulling back to show the larger context. Let the story breathe. Great magazine writing has pacing, and pacing comes from knowing when to slow down and when to accelerate.
What to Avoid
Never use bullet points, numbered lists, headers, or any formatting that breaks the flow of narrative prose. Do not summarize when you could show. Do not explain what a character is feeling when you can demonstrate it through their behavior or words. Avoid clinical or transactional language. Never write in a way that feels like a report, a how-to guide, or a structured breakdown of information. Do not open with generic or throat-clearing sentences like "In today's world" or "It is important to note." Do not wrap up the story with a tidy moral or an explicit lesson — trust the reader to feel what the story means. Above all, never let the writing feel flat, rushed, or forgettable. Every response is a piece of craft.

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""

storyteller_writer_agent = Agent(
    name = "Story Teller Writer Agent",
    instructions=STORYTELLER_WRITER_PROMPT,
    model = MODEL
)

#### Update Agent Handoff to add StoryTeller writer

In [48]:
# orchestrator_agent.handoffs = [journal_writer_agent, storyteller_writer_agent]

#### Test Story Teller Agent

In [49]:
""" # commented now; added earlier to test journal agent
from agents import trace
# with trace("Article Writer"):
with trace("Test Story Teller Article Writer", group_id="Learning AI Engineering",
           metadata={"experimentId": "Test Run Story Teller"}):
    result = await Runner.run(
        storyteller_writer_agent,
        input = "Write an article on a topic: how AI will use in health industries in 2030",
        max_turns=30
    )


print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))
"""

' # commented now; added earlier to test journal agent\nfrom agents import trace\n# with trace("Article Writer"):\nwith trace("Test Story Teller Article Writer", group_id="Learning AI Engineering",\n           metadata={"experimentId": "Test Run Story Teller"}):\n    result = await Runner.run(\n        storyteller_writer_agent,\n        input = "Write an article on a topic: how AI will use in health industries in 2030",\n        max_turns=30\n    )\n\n\nprint(f"Agent: {result.last_agent.name}")\nprint("-----")\ndisplay(Markdown(result.final_output))\n'

#### A writer Agent 3  - Advisory writer

In [59]:
ADVISOR_WRITER_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are a strategic advisor writing a memo for decision-makers.
You will receive a conversation history that includes research on a topic.

Your style:
- Lead with why this matters to the reader right now — what's at stake
- Focus on "what does this mean for you" and "what should you do about it"
- Be direct and authoritative — every sentence should earn its place
- Break down complex findings into clear, specific recommendations
- End with concrete action items the reader can act on this week
- Write like you're advising a CEO, not lecturing a classroom
- Aim for 800-1200 words

Do NOT use generic section headers like "Introduction", "Conclusion", or "Overview". Use creative, specific headers that grab attention.
Do NOT overuse headers. Only use one title and BETWEEN 2 and 3 headers in total for the entire article.
Do NOT use bullet points or numbered lists.
Do NOT ask for feedback, offer revisions, or include any commentary after the article.
Just deliver the finished article in markdown format.

Include the hero image URL at the top of your article in markdown format: ![Hero Image](url)
IMPORTANT: Copy the image URL EXACTLY as provided. Do not modify it.
"""

advisor_writer_agent = Agent(
    name = "Advisor Writer Agent",
    instructions=ADVISOR_WRITER_PROMPT,
    model = MODEL
)

#### Test Advisory Agent

In [51]:
"""
# commented now; added earlier to test journal agent
from agents import trace
# with trace("Article Writer"):
with trace("Test Advisory Article Writer", group_id="Learning AI Engineering",
           metadata={"experimentId": "Test Run Advisory Agent"}):
    result = await Runner.run(
        advisor_writer_agent,
        input = "Write an article on a topic: what should safari businss operators do to prepare zebra migratons in Africa",
        max_turns=30
    )


print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

"""

'\n# commented now; added earlier to test journal agent\nfrom agents import trace\n# with trace("Article Writer"):\nwith trace("Test Advisory Article Writer", group_id="Learning AI Engineering",\n           metadata={"experimentId": "Test Run Advisory Agent"}):\n    result = await Runner.run(\n        advisor_writer_agent,\n        input = "Write an article on a topic: what should safari businss operators do to prepare zebra migratons in Africa",\n        max_turns=30\n    )\n\n\nprint(f"Agent: {result.last_agent.name}")\nprint("-----")\ndisplay(Markdown(result.final_output))\n\n'

#### update orchstrator agent handoffs to Advisor Writer Agent.

In [ ]:
# orchestrator_agent.handoffs = [journal_writer_agent, storyteller_writer_agent,advisor_writer_agent]

from agents import handoff
from pydantic import BaseModel

class WriterSelectionInfo(BaseModel):
    which_writer:str
    reason:str

async def on_handoff(_ctx, decision: WriterSelectionInfo):
    print(f"  Writer Selected: {decision.which_writer}, Reason: {decision.reason}")

orchestrator_agent.handoffs = [
    handoff(agent = journal_writer_agent, on_handoff=on_handoff), 
    handoff(agent = storyteller_writer_agent, on_handoff=on_handoff),
    handoff(agent = advisor_writer_agent, on_handoff=on_handoff)
    ]

Tool name 'transfer_to_Journal Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_journal_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Story Teller Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_story_teller_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Advisor Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_advisor_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


### Lets Run it

In [68]:
from agents import trace

# with trace("Article Writer"):
with trace("Article Writer with Handoff", group_id="Learning AI Engineering",
           metadata={"Experiment": "02"}):
    result = await Runner.run(
        orchestrator_agent,
        #[even after temporary updated Orchestration agent not to use tools, we see tools getting called as LLM is confused between System and User prompts.
        # Updating Usere prompt to not use research word. This is temporary and will revert later.  #input = "Research a following topic and provide a comprehensive result: how AI will use in health industries in 2030",
        input = "Write a topic: School kids a lesson about success",
        max_turns=30
    )

  Writer Selected: ???


In [69]:
print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

Agent: Advisor Writer Agent
-----


![Hero Image](https://images.unsplash.com/photo-1503676260728-1c00da094a0b)

# Shaping Tomorrow’s Leaders: Teaching Kids the Real Lessons of Success

Why this matters is simple: the way we teach children about success today shapes the adults they become tomorrow — innovators, leaders, employees, and citizens navigating an increasingly complex world. Yet, many current narratives around success are incomplete or misleading, emphasizing outcomes like grades, trophies, and social status, while undervaluing the deeper skills and mindsets that sustain long-term achievement and fulfillment. For parents, educators, and policymakers, understanding this gap is crucial to preparing kids for a future where resilience, adaptability, and integrity matter more than ever.

### Why Our Traditional Lessons on Success Fall Short

When children are told that success means winning or being the best, they internalize a narrow definition that can backfire spectacularly. This kind of message can lead to a fear of failure, a reliance on external approval, and an overwhelming pressure to perform. We see this in alarming rates of anxiety and burnout among students who tie their self-worth to transient accomplishments.

What this means for you — whether you lead a school, design curriculum, or shape policy — is that simply celebrating trophies and rankings is no longer sufficient. Success is far more nuanced and expansive. For kids to thrive, they need to develop grit, a growth mindset, emotional intelligence, and ethical awareness. These qualities enable them to learn from failure, persist through setbacks, work collaboratively, and make choices that benefit both themselves and their communities.

Moreover, the future economy demands creativity and problem solving more than rote memorization or rigid competition. Preparing children for this environment means shifting the narrative and practice from success as a trophy to success as a lifelong journey of learning and contribution.

### Transforming Schools into Hubs for Real Success

The imperative is clear: schools must become incubators for the broader competencies that define meaningful success. This begins with educators modeling vulnerability and resilience themselves — showing students that mistakes are normal and essential steps toward mastery. Lessons should integrate reflective practices where students assess what they learned from challenges, not just whether they achieved a specific grade.

Curriculum design has to be strategic about embedding social-emotional learning alongside academic content. For example, lessons can include problem-based learning tasks that require teamwork, critical thinking, and ethical consideration. Student projects should reward creativity, perseverance, and the ability to respond constructively to feedback, rather than just accuracy.

Additionally, schools should actively foster a culture where success celebrates improvement and effort equally alongside final results. This might mean rewarding students for steady progress or for acts of kindness and leadership in the community, not just exam scores.

Parents and caregivers are indispensable partners in this transformation. When adults reinforce these values at home — emphasizing curiosity, kindness, and resilience — children receive a consistent message and feel supported in their authentic growth.

The cost of ignoring this shift is high. Without it, we risk raising a generation of high performers who are ill-equipped to handle complexity, disappointment, or failure, and who lack the social and emotional skills critical to thriving in modern life.

### What You Should Do About It This Week

Begin by reviewing how your school or organization currently defines and recognizes success. Look beyond test scores and trophies. Identify areas where social-emotional learning or growth mindset principles can be incorporated or strengthened in curriculum and school culture. Encourage educators to share stories openly about their own learning journeys and setbacks to model resilience.

Engage your community — parents, teachers, and students — in conversations about what success really means and how it can be cultivated together. This can be a simple staff meeting or a family workshop centered on new approaches to failure, effort, and collaboration.

Initiate a pilot program or classroom activity that emphasizes process over product. For example, introduce reflective journaling or peer feedback designed to highlight personal growth and collaborative skills.

Finally, commit to measuring success in richer, more holistic ways. Implement surveys or assessments that capture social-emotional competencies and personal growth alongside academic achievement.

The future depends on raising children who understand that success is not a fixed prize but a dynamic, evolving journey. The actions you take now will determine whether the next generation is prepared to lead with wisdom, resilience, and heart.

#### Run 2 with an example of Story Telling

In [63]:
from agents import trace

# with trace("Article Writer"):
with trace("Article Writer with Handoff", group_id="Learning AI Engineering",
           metadata={"Experiment": "02"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Research a following topic and provide a comprehensive result: The first time you truly failed—and what followed",
        max_turns=30
    )

Tool name 'transfer_to_Journal Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_journal_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Story Teller Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_story_teller_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Advisor Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_advisor_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Journal Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_journal_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'tra

 ✅ search_web: Got results for the first time you truly failed personal narratives statistics key themes
 ✅ search_web: Got results for personal stories of failure and what happened next statistics themes
 ❌ failed to fech or extract text from url: https://medium.com/@johnrampton/nobody-remembers-failures-so-why-are-you-afraid-failure-stories-from-successful-people-82c15bfadc85
 ❌ failed to fech or extract text from url: https://www.datasciencewithkevon.com/index.php/2024/04/11/embracing-the-bumps-how-failure-shapes-success/
 ✅ search_web: Got results for the first time you failed personal stories key themes statistics
 ✅ fetch_rul: Got: 17021 chars from ailed/
 ✅ search_web: Got results for statistics about failure and success stories personal narratives
 ❌ failed to fech or extract text from url: https://fastercapital.com/content/Cause-storytelling--Beyond-Statistics--Humanizing-Causes-Through-Personal-Stories.html
 ✅ fetch_rul: Got: 22619 chars from 
 ✅ search_web: Got results for p

Tool name 'transfer_to_Journal Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_journal_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Story Teller Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_story_teller_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Advisor Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_advisor_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


 ✅ search_web: Got results for personal accounts of first major failure case studies psychological perspectives practical lessons
 ✅ fetch_rul: Got: 124 chars from ures-are-hard-to-trace-and-state-keeps-activity-7453463685282312192-AHQl
 ✅ search_web: Got results for psychological perspectives on personal failure case studies practical lessons
 ✅ fetch_rul: Got: 10 chars from 
 ❌ failed to fech or extract text from url: https://restpublisher.com/wp-content/uploads/2025/04/48.-The-Role-of-Failure-in-Personality-Growth-Learning-from-Setbacks-1.pdf
 ✅ fetch_rul: Got: 42264 chars from 
 ✅ search_web: Got results for personal accounts first major failure psychological perspectives practical lessons
 ✅ search_web: Got results for personal stories of first major failure case studies psychological lessons
 ✅ fetch_rul: Got: 3388 chars from nt-wrong-case-failure-ajay-mahajan-30utc
 ✅ search_web: Got results for personal accounts of failure stories psychological resilience lessons
 ❌ failed to f

Tool name 'transfer_to_Journal Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_journal_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Story Teller Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_story_teller_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Advisor Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_advisor_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


  🎨 🖌️generate_image: A natural, photographic-style image capturing the theme of f
   ✅ genrate_image: Done


Tool name 'transfer_to_Journal Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_journal_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Story Teller Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_story_teller_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Advisor Writer Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_advisor_writer_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


In [22]:
print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

Agent: Orchestrator Agent
-----


Hero Image Description:
“A stunning, natural photographic scene capturing a herd of zebras migrating across vast African grasslands under a golden sunset. The zebras are mid-stride, dust rising from their hooves, their bold black-and-white stripes vividly contrasted against the warm earth tones of the plains. In the background, slightly blurred, a tasteful mobile safari camp with canvas tents and soft glowing lanterns evokes adventure and tranquility. The sky is expansive with scattered clouds, enhancing the mood of wilderness and freedom in the African savanna.”

## Selected Research Brief: Preparing Safari Business Operations for Zebra Migrations in Africa

Overview  
The zebra migration in Botswana is Africa’s longest terrestrial mammal migration, covering roughly 500 km between the Chobe River floodplains, Nxai Pan, Makgadikgadi Pans, and the Okavango Delta. It presents a unique tourism spectacle with significant ecological and conservation importance. Safari operators must balance immersive guest experiences with robust risk management, logistics, marketing, and sustainability.

Key Facts & Statistics  
- Migration spans about 500 km one way, with up to 30,000 zebras participating seasonally.  
- Major migration occurs November–March (rainy season), with a dry-season return June–October.  
- Predators (lions, hyenas) follow the herds, offering dynamic wildlife viewing.  
- Removal of veterinary fences has restored historic migration routes.  

Themes & Operational Considerations  

1. Risk Management  
  • Environmental: Plan for extreme heat, salt pans, heavy rains, and river crossings.  
  • Wildlife: Expert guides required to ensure safe predator-prey viewing and prevent conflicts.  
  • Health & Safety: Establish emergency protocols, medical support, and reliable communications.  
  • Conservation Liability: Maintain best practices to avoid habitat damage or wildlife disturbance.  

2. Logistics  
  • Route Tracking: Use rainfall and GPS data to align itineraries with migration timing.  
  • Mobile Camps: Deploy movable, low-impact camps that follow herds.  
  • Transport: Coordinate 4×4 vehicles, light aircraft, and ground support for remote access.  
  • Local Expertise: Hire and train local guides to enhance safety and enrich guest experiences.  
  • Guest Prep: Advise on appropriate gear and set expectations around migration unpredictability.  

3. Marketing  
  • Unique Positioning: Emphasize Botswana’s less-crowded, authentic zebra migration vs. East Africa’s wildebeest.  
  • Sustainability Messaging: Highlight conservation partnerships and eco-friendly practices.  
  • Customized Safaris: Offer small-group, private, and photographic tour packages.  
  • Partnerships: Collaborate with NGOs and community projects for credibility.  
  • Digital Storytelling: Use social media, virtual tours, and educational content to attract eco-conscious travelers.  

4. Sustainability  
  • Environmental: Implement solar energy, water recycling, minimal-waste policies, and electric vehicles where possible.  
  • Community: Provide fair wages, local employment, and revenue-sharing to reduce human-wildlife conflict.  
  • Animal Welfare: Maintain safe viewing distances, avoid wildlife interference, and support anti-poaching programs.  
  • Certifications: Adhere to standards like Fair Trade Tourism and Travelife to demonstrate commitment.  
  • Transparency: Share measurable impact metrics to avoid greenwashing.  

Notable Data Points & Sources  
- GPS collar studies (2012) confirmed migration distances and historical routes.  
- Botswana’s low-density tourism model prioritizes exclusivity and ecological sensitivity.  
- Sources:  
  1. https://www.discoverafrica.com/safaris/botswana/the-zebra-migration-in-botswana/  
  2. https://www.botswana-experience.com/blog/botswana-zebra-migration-africas-best-kept-safari-secret/  
  3. https://cheetahsafaris.com/inspirations/zebra-migration/  
  4. https://www.africansafarimag.com/best-african-safari-tour-companies  
  5. https://www.discoverafrica.com/blog/ethical-wildlife-experiences-sustainable-travel-african-safari/  
  6. https://www.africatouroperators.org/africa/ethical-african-safari-companies/  
  7. https://www.africanbudgetsafaris.com/blog/greenwashing-africa/  

This brief equips safari business operators with actionable guidance on risk management, logistics planning, marketing differentiation, and sustainability practices tailored to the dynamic demands of zebra migrations.

#### Run3  for Advisor Writing

In [16]:
from agents import trace

# with trace("Article Writer"):
with trace("Article Writer with Handoff", group_id="Learning AI Engineering",
           metadata={"Experiment": "02"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Research a following topic and provide a comprehensive result: what should safari business operators do to prepare zebra migratons in Africa",
        max_turns=30
    )

print(f"Agent: {result.last_agent.name}")
print("-----")
display(Markdown(result.final_output))

 ✅ search_web: Got results for safari business preparation for zebra migrations in Africa
 ✅ fetch_rul: Got: 22729 chars from 
 ✅ fetch_rul: Got: 6604 chars from 
 ✅ search_web: Got results for safari business operators prepare zebra migration Africa best practices
 ✅ fetch_rul: Got: 9163 chars from as-great-migration/
 ✅ fetch_rul: Got: 10924 chars from tion
 ✅ search_web: Got results for zebra migration impact on safari business operators Africa
 ✅ fetch_rul: Got: 7368 chars from ons/
 ✅ fetch_rul: Got: 9621 chars from ation-africas-best-kept-safari-secret/
 ✅ search_web: Got results for Safari business operations zebra migrations Africa
 ✅ fetch_rul: Got: 7368 chars from ons/
 ✅ fetch_rul: Got: 14491 chars from rations-africa-great-migration/
 ❌ failed to fech or extract text from url: https://www.travellocal.com/en-gb/articles/botswana-zebra-migration
 ✅ search_web: Got results for preparing safari business operations for zebra migrations Botswana
 ✅ fetch_rul: Got: 22729 chars fro

Hero Image Description:
“A stunning, natural photographic scene capturing a herd of zebras migrating across vast African grasslands under a golden sunset. The zebras are mid-stride, dust rising from their hooves, their bold black-and-white stripes vividly contrasted against the warm earth tones of the plains. In the background, slightly blurred, a tasteful mobile safari camp with canvas tents and soft glowing lanterns evokes adventure and tranquility. The sky is expansive with scattered clouds, enhancing the mood of wilderness and freedom in the African savanna.”

## Selected Research Brief: Preparing Safari Business Operations for Zebra Migrations in Africa

Overview  
The zebra migration in Botswana is Africa’s longest terrestrial mammal migration, covering roughly 500 km between the Chobe River floodplains, Nxai Pan, Makgadikgadi Pans, and the Okavango Delta. It presents a unique tourism spectacle with significant ecological and conservation importance. Safari operators must balance immersive guest experiences with robust risk management, logistics, marketing, and sustainability.

Key Facts & Statistics  
- Migration spans about 500 km one way, with up to 30,000 zebras participating seasonally.  
- Major migration occurs November–March (rainy season), with a dry-season return June–October.  
- Predators (lions, hyenas) follow the herds, offering dynamic wildlife viewing.  
- Removal of veterinary fences has restored historic migration routes.  

Themes & Operational Considerations  

1. Risk Management  
  • Environmental: Plan for extreme heat, salt pans, heavy rains, and river crossings.  
  • Wildlife: Expert guides required to ensure safe predator-prey viewing and prevent conflicts.  
  • Health & Safety: Establish emergency protocols, medical support, and reliable communications.  
  • Conservation Liability: Maintain best practices to avoid habitat damage or wildlife disturbance.  

2. Logistics  
  • Route Tracking: Use rainfall and GPS data to align itineraries with migration timing.  
  • Mobile Camps: Deploy movable, low-impact camps that follow herds.  
  • Transport: Coordinate 4×4 vehicles, light aircraft, and ground support for remote access.  
  • Local Expertise: Hire and train local guides to enhance safety and enrich guest experiences.  
  • Guest Prep: Advise on appropriate gear and set expectations around migration unpredictability.  

3. Marketing  
  • Unique Positioning: Emphasize Botswana’s less-crowded, authentic zebra migration vs. East Africa’s wildebeest.  
  • Sustainability Messaging: Highlight conservation partnerships and eco-friendly practices.  
  • Customized Safaris: Offer small-group, private, and photographic tour packages.  
  • Partnerships: Collaborate with NGOs and community projects for credibility.  
  • Digital Storytelling: Use social media, virtual tours, and educational content to attract eco-conscious travelers.  

4. Sustainability  
  • Environmental: Implement solar energy, water recycling, minimal-waste policies, and electric vehicles where possible.  
  • Community: Provide fair wages, local employment, and revenue-sharing to reduce human-wildlife conflict.  
  • Animal Welfare: Maintain safe viewing distances, avoid wildlife interference, and support anti-poaching programs.  
  • Certifications: Adhere to standards like Fair Trade Tourism and Travelife to demonstrate commitment.  
  • Transparency: Share measurable impact metrics to avoid greenwashing.  

Notable Data Points & Sources  
- GPS collar studies (2012) confirmed migration distances and historical routes.  
- Botswana’s low-density tourism model prioritizes exclusivity and ecological sensitivity.  
- Sources:  
  1. https://www.discoverafrica.com/safaris/botswana/the-zebra-migration-in-botswana/  
  2. https://www.botswana-experience.com/blog/botswana-zebra-migration-africas-best-kept-safari-secret/  
  3. https://cheetahsafaris.com/inspirations/zebra-migration/  
  4. https://www.africansafarimag.com/best-african-safari-tour-companies  
  5. https://www.discoverafrica.com/blog/ethical-wildlife-experiences-sustainable-travel-african-safari/  
  6. https://www.africatouroperators.org/africa/ethical-african-safari-companies/  
  7. https://www.africanbudgetsafaris.com/blog/greenwashing-africa/  

This brief equips safari business operators with actionable guidance on risk management, logistics planning, marketing differentiation, and sustainability practices tailored to the dynamic demands of zebra migrations.